# Trace Synthesizer: CFG-independent LSTM Training & Metrics

This notebook demonstrates how to load traces, visualize CFGs with trace overlays, and evaluate trace synthesis metrics (KL Divergence, Hot-Path Accuracy, etc.).

In [ ]:
%matplotlib inline
from __future__ import annotations

import json
from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from trace_synthesizer.core.grammar import CfgProgram
from trace_synthesizer.io.compress_pipeline import load_compressed_trace_json
from trace_synthesizer.viz.graphviz_renderer import CfgGraphvizRenderer
from trace_synthesizer.metrics.loaders import load_path_from_intra_trace_json, load_paths_from_intra_traces_jsonl
from trace_synthesizer.metrics.compare import run_metrics, DEFAULT_METRIC_ORDER
from trace_synthesizer.metrics.types import MetricContext

## 1. Visualizing the CFG and Ground-Truth Trace

In [ ]:
REPO = Path("..").resolve()
CFG_PATH = REPO / "output" / "production_validation" / "benchmark_complex.cfg.json"
TRACE_PATH = REPO / "output" / "production_validation" / "reference_real_intra.json"
FUNC = "main"

if CFG_PATH.is_file() and TRACE_PATH.is_file():
    grammar = CfgProgram.from_cfg_json(CFG_PATH)
    fn = grammar.function(FUNC)
    
    # Load trace for overlay
    with open(TRACE_PATH) as f:
        data = json.load(f)
    trace_bb = [int(bb["bb"]) for bb in data.get("sequence", []) if bb.get("func") == FUNC]
    
    renderer = CfgGraphvizRenderer(fn, trace_for_func=trace_bb, graph_format="svg")
    renderer.render("benchmark_complex_main_overlay")
    print("Graph exported to benchmark_complex_main_overlay.svg")
    
    # Display in notebook if possible
    try:
        from IPython.display import SVG, display
        display(SVG("benchmark_complex_main_overlay.svg"))
    except ImportError:
        pass
else:
    print(f"Run production validation first to generate {CFG_PATH}")

## 2. Evaluate Metrics: Baseline vs LSTM

In [ ]:
# Compare RandomPGO rollouts vs LSTM rollouts vs Ground Truth
OUT_DIR = REPO / "output" / "production_validation"
RANDOM_TRACES = OUT_DIR / "rollouts_random" / "intra_traces.jsonl"
LSTM_TRACES = OUT_DIR / "rollouts_lstm" / "intra_traces.jsonl"

if TRACE_PATH.is_file() and RANDOM_TRACES.is_file() and LSTM_TRACES.is_file():
    ref_paths = [load_path_from_intra_trace_json(TRACE_PATH)]
    random_paths = load_paths_from_intra_traces_jsonl(RANDOM_TRACES)
    lstm_paths = load_paths_from_intra_traces_jsonl(LSTM_TRACES)
    
    ctx = MetricContext(function_name=FUNC, epsilon=1e-8, ngram_min=2, ngram_max=4, top_k=64)
    
    res_random = run_metrics(ref_paths, random_paths, ctx, names=DEFAULT_METRIC_ORDER)
    res_lstm = run_metrics(ref_paths, lstm_paths, ctx, names=DEFAULT_METRIC_ORDER)
    
    metrics = ["block_visit_kl", "edge_transition_kl", "hot_path_ngram_overlap"]
    random_vals = [res_random[m].value for m in metrics]
    lstm_vals = [res_lstm[m].value for m in metrics]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(metrics))
    width = 0.35
    
    ax.bar([i - width/2 for i in x], random_vals, width, label='RandomPGO Baseline')
    ax.bar([i + width/2 for i in x], lstm_vals, width, label='Feature-Window LSTM')
    
    ax.set_ylabel('Score')
    ax.set_title('Evaluation Metrics: Baseline vs LSTM')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Rollout traces not found. Run production validation script first.")